In [ ]:
# Import necessary libraries
import os
import nest_asyncio
#import requests
# Import strands components
from strands import Agent, tool
from strands.models.ollama import OllamaModel

# Allow nested event loops in Jupyter notebooks
nest_asyncio.apply()

In [ ]:
# Ollama 모델 구성
ollama_model = OllamaModel(
    model_id="llama3.2:latest",
    host="http://localhost:11434",
    params={
        "max_tokens": 2048,
        "temperature": 0.7,
        "top_p": 0.9,
        "stream": True,
    },
)

In [ ]:
# File Operation Tools
@tool
def file_read(file_path: str) -> str:
    """Read a file and return its content. Supports both text and PDF files.

    Args:
        file_path (str): Path to the file to read

    Returns:
        str: Content of the file

    Raises:
        FileNotFoundError: If the file doesn't exist
    """
    try:
        # Check if it's a PDF file
        if file_path.lower().endswith('.pdf'):
            import PyPDF2
            with open(file_path, "rb") as file:
                pdf_reader = PyPDF2.PdfReader(file)
                text = ""
                for page in pdf_reader.pages:
                    text += page.extract_text() + "\n"
                return text if text.strip() else "Error: Could not extract text from PDF"
        else:
            # Regular text file
            with open(file_path, "r", encoding="utf-8") as file:
                return file.read()
    except FileNotFoundError:
        return f"Error: File '{file_path}' not found."
    except Exception as e:
        return f"Error reading file: {str(e)}"


@tool
def file_write(file_path: str, content: str) -> str:
    """Write content to a file.

    Args:
        file_path (str): The path to the file
        content (str): The content to write to the file

    Returns:
        str: A message indicating success or failure
    """
    try:
        # Create directory if it doesn't exist
        os.makedirs(os.path.dirname(os.path.abspath(file_path)), exist_ok=True)

        with open(file_path, "w") as file:
            file.write(content)
        return f"File '{file_path}' written successfully."
    except Exception as e:
        return f"Error writing to file: {str(e)}"


@tool
def list_directory(directory_path: str = ".") -> str:
    """List files and directories in the specified path.

    Args:
        directory_path (str): Path to the directory to list

    Returns:
        str: A formatted string listing all files and directories
    """
    try:
        items = os.listdir(directory_path)
        files = []
        directories = []

        for item in items:
            full_path = os.path.join(directory_path, item)
            if os.path.isdir(full_path):
                directories.append(f"Folder: {item}/")
            else:
                files.append(f"File: {item}")

        result = f"Contents of {os.path.abspath(directory_path)}:\n"
        result += (
            "\nDirectories:\n" + "\n".join(sorted(directories))
            if directories
            else "\nNo directories found."
        )
        result += (
            "\n\nFiles:\n" + "\n".join(sorted(files)) if files else "\nNo files found."
        )

        return result
    except Exception as e:
        return f"Error listing directory: {str(e)}"

In [ ]:
system_prompt="You are a helpful AI assistant."

# 도구를 가진 에이전트 생성
local_agent = Agent(
    system_prompt=system_prompt,
    model=ollama_model,
    tools=[file_read, file_write, list_directory],
)

In [ ]:
local_agent("현재 디렉토리의 파일 목록 출력")

In [ ]:
local_agent.tool.list_directory()

In [ ]:
local_agent("현재 디렉토리의 파일과 디렉토리 목록 출력")

In [ ]:
local_agent("위치를 /home/sagemaker-user 으로 변경하고 파일과 디렉토리 목록 알려줘")

In [ ]:
local_agent.tool.list_directory(directory_path = "/home/sagemaker-user")